# Query the Concept Paper (ChromaDB)

This notebook lets you run semantic search over the concept-paper chapters that were ingested
into a persistent ChromaDB collection by `tools/ingest_concept_paper.py`.

**Before running:** select the project virtual environment (`.venv`) as the notebook kernel so
that `chromadb` and the ingestion module are importable.

If the collection is empty, the setup cell below will ingest the chapters for you.

In [3]:
import sys
from pathlib import Path

# Make the repo root importable so we can reuse the ingestion module.
REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "tools" / "ingest_concept_paper.py").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from tools.ingest_concept_paper import get_collection, ingest, COLLECTION_NAME, DEFAULT_DB_DIR

print("Repo root :", REPO_ROOT)
print("DB dir    :", DEFAULT_DB_DIR)
print("Collection:", COLLECTION_NAME)

Repo root : /Users/zdenek.strbik/IdeaProjects/workspace-apex/databricks-bootcamp-labs
DB dir    : /Users/zdenek.strbik/IdeaProjects/workspace-apex/databricks-bootcamp-labs/.chroma
Collection: concept_paper


In [5]:
# Connect to the persistent collection; ingest on first run if it is empty.
collection = get_collection()
if collection.count() == 0:
    print("Collection is empty - ingesting chapters...")
    ingest()
    collection = get_collection()

print(f"Collection '{collection.name}' holds {collection.count()} chunks.")

Collection 'concept_paper' holds 1359 chunks.


In [7]:
from textwrap import shorten


def search(query, n_results=5, where=None):
    """Run a semantic search and pretty-print the top matches.

    Parameters
    ----------
    query : str
        Natural-language question or keywords.
    n_results : int
        Number of chunks to return.
    where : dict | None
        Optional ChromaDB metadata filter, e.g. {"chapter": "6"}.
    """
    res = collection.query(query_texts=[query], n_results=n_results, where=where)
    docs = res["documents"][0]
    metas = res["metadatas"][0]
    dists = res["distances"][0]
    print(f"Query: {query!r}  (filter={where})\n" + "=" * 88)
    for rank, (doc, meta, dist) in enumerate(zip(docs, metas, dists), start=1):
        score = 1 - dist  # cosine distance -> similarity
        print(f"\n[{rank}] similarity={score:.3f}  |  {meta.get('title', '(no title)')}")
        print(f"      source : {meta.get('source')}")
        if meta.get("heading"):
            print(f"      heading: {meta['heading']}")
        snippet = " ".join(doc.split())
        print("      " + shorten(snippet, width=400, placeholder=" ..."))
    return res

## Ask a question

Change the text below to query the concept paper.

In [8]:
_ = search("How is the PD calibration target calculated?", n_results=5)

Query: 'How is the PD calibration target calculated?'  (filter=None)

[1] similarity=0.628  |  6.6 Calibration to the calibration target
      source : 06_pd_calibration/06_calibration_to_the_calibration_target.md
      heading: 6.6 Calibration to the calibration target
      6.2, related to Micro/SME rating system. Prior to detailing the PD calibration procedure, a brief overview is provided on maintaining the stability of PD estimates and the control measures set for this purpose. To ensure the stability of PD estimates, keeping the calibrated portfolio PD before MoC aligned with the calibration target (CT) until the next annual re-estimation, developers should ...

[2] similarity=0.621  |  6.5 Calibration target calculation
      source : 06_pd_calibration/05_calibration_target_calculation.md
      [[00_index|🏠 Index]] › [[06_pd_calibration/00_overview|6 PD Calibration]] # 6.5 Calibration target calculation 6.5.1 Calculation of the 1-Y default rate Calibration target calculation inv

In [ ]:
_ = search("What are the margins of conservatism and when do they apply?", n_results=5)

## Filter by chapter

Metadata carries the `chapter` number (as a string) and `tags`, so you can scope a search to a
single chapter. For example, restrict results to chapter 6 (PD calibration):

In [ ]:
_ = search("representativeness analysis of the calibration data", n_results=5, where={"chapter": "6"})

## Re-ingest after editing the chapters

If the markdown chapters change, re-run the ingestion (either from a terminal with
`python tools/ingest_concept_paper.py --reset`, or with the cell below) and reconnect.

In [ ]:
# ingest(reset=True)
# collection = get_collection()
# collection.count()